In [0]:
from pyspark.sql import functions as F

events = spark.table("events_delta")

features_df = events.groupBy("user_id").agg(
    F.count("*").alias("total_events"),
    F.count(F.when(F.col("event_type")=="purchase",1)).alias("purchases"),
    F.sum("price").alias("total_spent"),
    F.avg("price").alias("avg_price")
)

In [0]:
features_df.display()

In [0]:
features_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("user_features")

In [0]:
features_df = spark.table("user_features")
features_df.display()

In [0]:
events = spark.table("events_delta")

events.display()

In [0]:
from pyspark.sql import functions as F

features_df = events.groupBy("user_id").agg(
    F.count("*").alias("total_events"),
    F.count(F.when(F.col("event_type")=="purchase",1)).alias("purchases"),
    F.sum("price").alias("total_spent"),
    F.avg("price").alias("avg_price")
)

features_df.display()

In [0]:
features_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("user_features")

In [0]:
spark.sql("SELECT * FROM user_features LIMIT 20").display()

In [0]:
label_df = events.groupBy("user_id").agg(
    F.max(
        F.when(F.col("event_type")=="purchase",1).otherwise(0)
    ).alias("purchased")
)

label_df.display()

In [0]:
training_data = features_df.join(label_df,"user_id")

training_data.display()

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "total_events",
        "purchases",
        "total_spent",
        "avg_price"
    ],
    outputCol="features"
)

ml_data = assembler.transform(training_data) \
    .select("features","purchased")

ml_data.display()

In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol="purchased",
    numTrees=20
)

final_model = rf.fit(ml_data)

In [0]:
predictions = final_model.transform(ml_data)

predictions.select(
    "features",
    "prediction",
    "purchased"
).display()

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="purchased"
)

auc = evaluator.evaluate(predictions)

print("Final Model AUC:", auc)

In [0]:
predictions.select(
    "prediction",
    "purchased"
).write.format("delta") \
 .mode("overwrite") \
 .saveAsTable("final_model_predictions")

In [0]:
spark.sql("""
SELECT *
FROM final_model_predictions
LIMIT 20
""").display()